In [4]:
# 3_test_research_design.ipynb
import os
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

BASE_DIR = r"C:\Users\User-NB\OneDrive\Desktop\ML"
DATA_PATH = os.path.join(BASE_DIR, "processed_data", "data_processed.pkl")
RESULT_DIR = os.path.join(BASE_DIR, "results")

# 1. 載入與切分
data = joblib.load(DATA_PATH)
X, y, fyear = data["X"], data["y"], data["fyear"]

train_mask = (fyear >= 2019) & (fyear <= 2023)
test_mask = (fyear == 2024)
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# 2. SMOTE
print("Training SMOTE Model...")
smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(penalty='l2', solver='liblinear', random_state=42))
])
smote_pipe.fit(X_train, y_train)
y_prob_smote = smote_pipe.predict_proba(X_test)[:, 1]

# 3. Calibration (Isotonic)
print("Applying Isotonic Calibration...")
# 這裡我們對 Weighted Baseline 進行校準
base_model = LogisticRegression(penalty='l2', class_weight='balanced', solver='liblinear', random_state=42)
calib_model = CalibratedClassifierCV(base_model, method='isotonic', cv=5)
calib_model.fit(X_train, y_train)
y_prob_calib = calib_model.predict_proba(X_test)[:, 1]

# 4. 存檔
joblib.dump({
    "y_prob_smote": y_prob_smote,
    "y_prob_calib": y_prob_calib
}, os.path.join(RESULT_DIR, "research_results.pkl"))
print("Research Design 完成。")

Training SMOTE Model...
Applying Isotonic Calibration...
Research Design 完成。
